# SKANN-SSL V3.3.0 Training — H100 Edition

**Version:** V3.3.0  
**Date:** January 2026  
**Platform:** Google Colab H100 80GB (Single GPU)

---

## V3.2 Root Cause Analysis → V3.3 Fixes

| V3.2 Bug | Impact | V3.3 Fix |
|----------|--------|----------|
| BT batch = 4 (grad accum doesn't help BT) | Loss never dropped (147→145) | **Physical batch = 64** |
| BatchNorm in projector with batch=4 | Garbage BN statistics | **LayerNorm** (batch-independent) |
| Fake SupCon phase (only sampling changed) | Confused training | **Removed** — pure SSL K=6/K=3 |
| LR scheduler hit zero at epoch 50 | No learning final epochs | **eta_min = 1e-6** |
| Pseudo-spectrogram 16×20000 (V2 was 16×4000) | 2D conv ineffective | **Added MaxPool1d(5)** → 16×4000 |
| AMP fp16 | Memory overhead | **bfloat16** (H100 native) |

---

## Expected Outcome

| Metric | V3.2 (Collapsed) | V3.3 Target |
|--------|------------------|-------------|
| Loss curve | 147 → 145 (flat) | 150 → **< 10** |
| Silhouette | -0.035 | **> 0.5** (target > 0.83) |
| Training time | ~3h (Kaggle T4x2) | ~1h (H100) |



## Cell 0: Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q umap-learn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 1: Configuration

**IMPORTANT:** Update `DATA_ROOT` to your Google Drive path where the V3.2 dataset is stored.

In [ ]:
import os

# =============================================================================
# CONFIGURATION — UPDATE THIS PATH
# =============================================================================
DATA_ROOT = "/content/drive/MyDrive/SKANN-SSL/v3_dataset"  # <-- UPDATE THIS

MANIFEST_PATH = os.path.join(DATA_ROOT, "master_dataset_manifest.csv")
PAIRING_PATH = os.path.join(DATA_ROOT, "pairing_manifest.csv")
TENSOR_DIR = os.path.join(DATA_ROOT, "tensors")

# Output directory
OUTPUT_DIR = "/content/drive/MyDrive/SKANN-SSL/v3_3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# V3.3 HYPERPARAMETERS
# =============================================================================
BATCH_SIZE = 64          # H100 can handle 64+. Try 32→64→96
EPOCHS = 50
BASE_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_EPOCHS = 3
BT_LAMBDA = 5e-3         # Barlow Twins redundancy reduction
LR_ETA_MIN = 1e-6        # Don't let LR hit zero!

# Architecture
SAMPLES_PER_CLIP = 80000  # 5 seconds @ 16kHz
LATENT_DIM = 128
SK_KERNEL_SIZES = (31, 63, 127, 255, 511, 1023)

# AMP with bfloat16 (H100 native)
USE_AMP = True
AMP_DTYPE = "bfloat16"

print(f"Batch size: {BATCH_SIZE}")
print(f"AMP: bfloat16 (H100 native)")
print(f"LR: {BASE_LR} → {LR_ETA_MIN} (never zero)")
print(f"Data root: {DATA_ROOT}")



## Cell 2: Validate Dataset

In [ ]:
import pandas as pd
import numpy as np

# Check files exist
assert os.path.exists(MANIFEST_PATH), f"Manifest not found: {MANIFEST_PATH}"
assert os.path.exists(PAIRING_PATH), f"Pairing not found: {PAIRING_PATH}"
assert os.path.exists(TENSOR_DIR), f"Tensors not found: {TENSOR_DIR}"

# Load manifests
manifest_df = pd.read_csv(MANIFEST_PATH)
pairing_df = pd.read_csv(PAIRING_PATH)

print(f"✅ Manifest: {len(manifest_df)} clips")
print(f"✅ Pairing: {len(pairing_df)} anchors")

# Class distribution
print(f"\nClass distribution:")
for cls, count in manifest_df['vessel_class'].value_counts().items():
    print(f"  {cls}: {count}")

# Check K per class
print(f"\nPartners per class:")
for cls in sorted(pairing_df['vessel_class'].unique()):
    grp = pairing_df[pairing_df['vessel_class'] == cls]
    avg_k = grp['partner_clip_ids'].apply(lambda x: len(str(x).split('|'))).mean()
    print(f"  {cls}: K={avg_k:.0f}")

# Check tensor count and shape
tensor_files = [f for f in os.listdir(TENSOR_DIR) if f.endswith('.npy')]
print(f"\n✅ Tensor files: {len(tensor_files)}")

# Verify shape consistency
sample = np.load(os.path.join(TENSOR_DIR, tensor_files[0]))
print(f"✅ Tensor shape: {sample.shape}")

if sample.shape != (1, 1, 80000):
    print(f"⚠️  Expected (1, 1, 80000), got {sample.shape}")

## Cell 3: Model Definition (LayerNorm Projector)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SKConv1DBlock(nn.Module):
    """Selective Kernel 1D Convolution with multiple kernel sizes."""
    
    def __init__(self, in_channels, out_channels, kernel_sizes, stride=1):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels, k, stride=stride, padding=k//2)
            for k in kernel_sizes
        ])
        # Attention
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(out_channels, out_channels // 4)
        self.fc2 = nn.Linear(out_channels // 4, len(kernel_sizes) * out_channels)
        self.softmax = nn.Softmax(dim=1)
        self.out_channels = out_channels
        self.n_branches = len(kernel_sizes)
    
    def forward(self, x):
        # Multi-branch convolutions
        branch_outs = [branch(x) for branch in self.branches]
        stacked = torch.stack(branch_outs, dim=1)  # [B, n_branches, C, T]
        
        # Fused features
        fused = sum(branch_outs)  # [B, C, T]
        
        # Attention weights
        gap = self.gap(fused).squeeze(-1)  # [B, C]
        attn = F.relu(self.fc1(gap))
        attn = self.fc2(attn)  # [B, n_branches * C]
        attn = attn.view(-1, self.n_branches, self.out_channels)  # [B, n_branches, C]
        attn = self.softmax(attn).unsqueeze(-1)  # [B, n_branches, C, 1]
        
        # Weighted sum
        out = (stacked * attn).sum(dim=1)  # [B, C, T]
        return out


class HybridSKEncoder(nn.Module):
    """
    V3.3 Encoder with LayerNorm in projector (not BatchNorm).
    
    Architecture:
    1. SK Conv1D backbone (multi-scale kernels)
    2. Reshape to pseudo-spectrogram
    3. 2D convolutions
    4. Global pooling
    5. Projector MLP with LayerNorm
    """
    
    def __init__(self, kernel_sizes=SK_KERNEL_SIZES, latent_dim=LATENT_DIM):
        super().__init__()
        
        # 1D Backbone with SK convolutions
        # Extra MaxPool1d(5) at end to match V2 pseudo-spectrogram shape
        # 80000 → /4 → /4 → /4 → /5 = 250 (same as V2's 16000/64)
        self.backbone1d = nn.Sequential(
            SKConv1DBlock(1, 64, kernel_sizes),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),           # 80000 → 20000
            
            SKConv1DBlock(64, 128, kernel_sizes),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(4),           # 20000 → 5000
            
            SKConv1DBlock(128, 256, kernel_sizes),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(4),           # 5000 → 1250
            
            nn.MaxPool1d(5),           # 1250 → 250 (V2-shaped!)
        )
        
        # 2D processing (pseudo-spectrogram)
        self.backbone2d = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        
        # Projector MLP with LAYERNORM (not BatchNorm!)
        self.projector = nn.Sequential(
            nn.Linear(256 * 4 * 4, 4096),
            nn.LayerNorm(4096),  # LayerNorm instead of BatchNorm
            nn.ReLU(),
            nn.Linear(4096, 4096),
            nn.LayerNorm(4096),  # LayerNorm instead of BatchNorm
            nn.ReLU(),
            nn.Linear(4096, latent_dim),
        )
    
    def forward(self, x):
        # x: [B, 1, 80000]
        
        # 1D backbone
        x = self.backbone1d(x)  # [B, 256, T']
        
        # Reshape to pseudo-spectrogram
        B, C, T = x.shape
        H = 16
        W = T * C // (H * 256)
        x = x.view(B, 1, H, -1)  # [B, 1, H, W]
        
        # 2D backbone
        x = self.backbone2d(x)  # [B, 256, 4, 4]
        
        # Flatten and project
        x = x.view(B, -1)  # [B, 256*4*4]
        x = self.projector(x)  # [B, latent_dim]
        
        return x


# Test model
model = HybridSKEncoder()
dummy = torch.randn(2, 1, 80000)
out = model(dummy)
print(f"✅ Model output shape: {out.shape}")
print(f"✅ Model parameters: {sum(p.numel() for p in model.parameters()):,}")


## Cell 4: Dataset (Pure SSL with K=6/K=3)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class SSLDataset(Dataset):
    """
    Pure SSL Dataset with hard positive mining (K=6 for vessels, K=3 for no_vessel).
    
    NO SupCon phase — just SSL with pairing manifest.
    Each epoch samples 1 random partner from the K available.
    """
    
    def __init__(self, manifest_path, pairing_path, tensor_dir):
        self.tensor_dir = tensor_dir
        
        # Load manifests
        self.df = pd.read_csv(manifest_path)
        pairing_df = pd.read_csv(pairing_path)
        
        # Build class mapping
        self.classes = sorted(self.df['vessel_class'].unique())
        self.class_to_id = {c: i for i, c in enumerate(self.classes)}
        
        # Build pairing lookup
        self.pairing_lookup = {}
        for _, row in pairing_df.iterrows():
            anchor_id = int(row['anchor_clip_id'])
            partner_ids = [int(x) for x in str(row['partner_clip_ids']).split('|')]
            self.pairing_lookup[anchor_id] = partner_ids
        
        print(f"[SSLDataset] Loaded {len(self.df)} clips, {len(self.pairing_lookup)} pairings")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip_id = int(row['clip_id'])
        vessel_class = row['vessel_class']
        label = self.class_to_id[vessel_class]
        
        # Load anchor
        anchor_path = os.path.join(self.tensor_dir, f"tensor_{clip_id:06d}.npy")
        anchor = np.load(anchor_path).astype(np.float32)
        
        # Get hard positive (random 1 of K)
        partner_ids = self.pairing_lookup.get(clip_id, [clip_id])
        pos_clip_id = int(np.random.choice(partner_ids))
        
        pos_path = os.path.join(self.tensor_dir, f"tensor_{pos_clip_id:06d}.npy")
        positive = np.load(pos_path).astype(np.float32)
        
        # Ensure shape [1, 80000]
        anchor = anchor.flatten().reshape(1, -1)
        positive = positive.flatten().reshape(1, -1)
        
        return torch.from_numpy(anchor), torch.from_numpy(positive), label


# Test dataset
dataset = SSLDataset(MANIFEST_PATH, PAIRING_PATH, TENSOR_DIR)
y1, y2, label = dataset[0]
print(f"✅ Anchor shape: {y1.shape}, Positive shape: {y2.shape}, Label: {label}")

## Cell 5: Barlow Twins Loss

In [ ]:
def barlow_twins_loss(z1, z2, lambd=BT_LAMBDA):
    """
    Barlow Twins loss.
    
    With proper batch size (16+), this should converge to < 10.
    """
    batch_size = z1.size(0)
    
    # Normalize along batch dimension
    z1_norm = (z1 - z1.mean(dim=0)) / (z1.std(dim=0) + 1e-6)
    z2_norm = (z2 - z2.mean(dim=0)) / (z2.std(dim=0) + 1e-6)
    
    # Cross-correlation matrix [D x D]
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    
    # Loss: on-diagonal (invariance) + off-diagonal (redundancy reduction)
    on_diag = torch.diagonal(c).add(-1).pow(2).sum()
    off_diag = c.flatten()[:-1].view(c.size(0)-1, c.size(0)+1)[:, 1:].flatten().pow(2).sum()
    
    return on_diag + lambd * off_diag


# Test loss
z1 = torch.randn(16, 128)  # Batch of 16
z2 = torch.randn(16, 128)
loss = barlow_twins_loss(z1, z2)
print(f"✅ Test loss (random embeddings): {loss.item():.2f}")

## Cell 6: Training Loop

In [ ]:
# H100 uses native bfloat16 via torch.autocast
from datetime import datetime
import time


def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on: {device}")
    
    # Model
    model = HybridSKEncoder().to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Dataset & Loader
    dataset = SSLDataset(MANIFEST_PATH, PAIRING_PATH, TENSOR_DIR)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True  # Important for BT batch statistics
    )
    print(f"Batches per epoch: {len(loader)}")
    print(f"Effective BT batch size: {BATCH_SIZE} (real, not accumulated!)")
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    
    # LR Scheduler: warmup then cosine with eta_min (never hits zero!)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=EPOCHS - WARMUP_EPOCHS,
        eta_min=LR_ETA_MIN
    )
    
    # Manual warmup for first few epochs
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1/WARMUP_EPOCHS,
        end_factor=1.0,
        total_iters=WARMUP_EPOCHS
    )
    
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, scheduler],
        milestones=[WARMUP_EPOCHS]
    )
    
    # AMP
    # H100 native bfloat16 - no scaler needed!
    # scaler = GradScaler() if USE_AMP else None
    
    # Loss history
    loss_history = []
    
    print(f"\n{'='*60}")
    print(f"SKANN-SSL V3.3.0 Training — Pure SSL, LayerNorm, Batch={BATCH_SIZE}")
    print(f"{'='*60}\n")
    
    start_time = time.time()
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        
        for step, (y1, y2, _) in enumerate(loader):
            y1 = y1.to(device)
            y2 = y2.to(device)
            
            optimizer.zero_grad()
            
            if USE_AMP:
                # H100 native bfloat16 - no scaler needed!
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    emb1 = model(y1)
                    emb2 = model(y2)
                    loss = barlow_twins_loss(emb1, emb2)
                
                loss.backward()
                optimizer.step()
            else:
                emb1 = model(y1)
                emb2 = model(y2)
                loss = barlow_twins_loss(emb1, emb2)
                loss.backward()
                optimizer.step()
            
            epoch_loss += loss.item()
        
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        current_lr = scheduler.get_last_lr()[0]
        
        loss_history.append((epoch, avg_loss, current_lr))
        
        # Progress logging
        elapsed = time.time() - start_time
        eta = elapsed / epoch * (EPOCHS - epoch)
        
        print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e} | ETA: {eta/60:.1f}m")
        
        # Save checkpoint every 10 epochs
        if epoch % 10 == 0:
            ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint_epoch{epoch}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, ckpt_path)
            print(f"  💾 Saved checkpoint: {ckpt_path}")
    
    # Save final model
    final_path = os.path.join(OUTPUT_DIR, "SKANN_SSL_V3_3_Final.pth")
    torch.save(model.state_dict(), final_path)
    print(f"\n✅ Final model saved: {final_path}")
    
    # Save loss history
    loss_path = os.path.join(OUTPUT_DIR, "loss_history.txt")
    with open(loss_path, 'w') as f:
        f.write("epoch,loss,lr\n")
        for e, l, lr in loss_history:
            f.write(f"{e},{l:.4f},{lr:.2e}\n")
    print(f"✅ Loss history saved: {loss_path}")
    
    total_time = time.time() - start_time
    print(f"\n⏱️  Total training time: {total_time/60:.1f} minutes")
    
    return model, loss_history


# Run training
model, loss_history = train()


## Cell 7: Extract Embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings(model):
    device = next(model.parameters()).device
    model.eval()
    
    # Load all clips (not pairs)
    df = pd.read_csv(MANIFEST_PATH)
    classes = sorted(df['vessel_class'].unique())
    class_to_id = {c: i for i, c in enumerate(classes)}
    
    all_embeddings = []
    all_labels = []
    all_clip_ids = []
    
    for idx in range(len(df)):
        row = df.iloc[idx]
        clip_id = int(row['clip_id'])
        label = class_to_id[row['vessel_class']]
        
        # Load tensor
        tensor_path = os.path.join(TENSOR_DIR, f"tensor_{clip_id:06d}.npy")
        x = np.load(tensor_path).astype(np.float32)
        x = x.flatten().reshape(1, 1, -1)  # [1, 1, 80000]
        x = torch.from_numpy(x).to(device)
        
        emb = model(x).cpu().numpy().squeeze()
        
        all_embeddings.append(emb)
        all_labels.append(label)
        all_clip_ids.append(clip_id)
        
        if (idx + 1) % 2000 == 0:
            print(f"  Extracted {idx + 1}/{len(df)}")
    
    embeddings = np.array(all_embeddings)
    labels = np.array(all_labels)
    clip_ids = np.array(all_clip_ids)
    
    print(f"\n✅ Embeddings shape: {embeddings.shape}")
    
    # Save
    np.save(os.path.join(OUTPUT_DIR, "embeddings_v3_3.npy"), embeddings)
    np.save(os.path.join(OUTPUT_DIR, "labels_v3_3.npy"), labels)
    np.save(os.path.join(OUTPUT_DIR, "clip_ids_v3_3.npy"), clip_ids)
    
    return embeddings, labels, clip_ids


embeddings, labels, clip_ids = extract_embeddings(model)

## Cell 8: Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.pyplot as plt

# Overall silhouette
sil_score = silhouette_score(embeddings, labels, metric='cosine')
print(f"\n{'='*60}")
print(f"SILHOUETTE SCORE: {sil_score:.4f}")
print(f"{'='*60}")

if sil_score > 0.8:
    print("✅ EXCELLENT — Better than V2 baseline!")
elif sil_score > 0.5:
    print("✅ GOOD — Clear class separation")
elif sil_score > 0.2:
    print("⚠️  WEAK — Some separation but not great")
elif sil_score > 0:
    print("⚠️  POOR — Marginal separation")
else:
    print("❌ FAILED — Worse than random")

# Per-class silhouette
sil_samples = silhouette_samples(embeddings, labels, metric='cosine')

df = pd.read_csv(MANIFEST_PATH)
classes = sorted(df['vessel_class'].unique())

print(f"\nPer-class silhouette:")
for i, cls in enumerate(classes):
    cls_mask = labels == i
    cls_sil = sil_samples[cls_mask].mean()
    print(f"  {cls}: {cls_sil:.4f}")

## Cell 9: UMAP Visualization

In [ ]:
import umap

# UMAP projection
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
umap_emb = reducer.fit_transform(embeddings)

# Plot
df = pd.read_csv(MANIFEST_PATH)
classes = sorted(df['vessel_class'].unique())
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))

fig, ax = plt.subplots(figsize=(12, 10))

for i, cls in enumerate(classes):
    mask = labels == i
    ax.scatter(umap_emb[mask, 0], umap_emb[mask, 1], 
               c=[colors[i]], label=cls, alpha=0.6, s=10)

ax.set_title(f'SKANN-SSL V3.3 Embeddings (Silhouette: {sil_score:.4f})', fontsize=14)
ax.legend(loc='best', fontsize=10)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')

plt.tight_layout()
umap_path = os.path.join(OUTPUT_DIR, "umap_v3_3.png")
plt.savefig(umap_path, dpi=150)
plt.show()

print(f"\n✅ UMAP saved: {umap_path}")

## Cell 10: Loss Curve

In [ ]:
epochs = [x[0] for x in loss_history]
losses = [x[1] for x in loss_history]
lrs = [x[2] for x in loss_history]

fig, ax1 = plt.subplots(figsize=(12, 6))

# Loss
ax1.plot(epochs, losses, 'b-', linewidth=2, label='Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Barlow Twins Loss', fontsize=12, color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.axhline(y=10, color='g', linestyle='--', alpha=0.5, label='Target (<10)')

# LR
ax2 = ax1.twinx()
ax2.plot(epochs, lrs, 'r-', linewidth=1, alpha=0.5, label='LR')
ax2.set_ylabel('Learning Rate', fontsize=12, color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_yscale('log')

ax1.set_title('V3.3 Training Loss & Learning Rate', fontsize=14)
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
loss_plot_path = os.path.join(OUTPUT_DIR, "loss_curve_v3_3.png")
plt.savefig(loss_plot_path, dpi=150)
plt.show()

print(f"\n✅ Loss curve saved: {loss_plot_path}")

## Cell 11: Summary

In [ ]:
print("\n" + "="*60)
print("V3.3 TRAINING COMPLETE")
print("="*60)
print(f"\nConfiguration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  AMP: {USE_AMP}")
print(f"  Projector: LayerNorm (not BatchNorm)")
print(f"  Strategy: Pure SSL (K=6/K=3)")

print(f"\nResults:")
print(f"  Final loss: {losses[-1]:.4f}")
print(f"  Silhouette: {sil_score:.4f}")

print(f"\nOutputs saved to: {OUTPUT_DIR}")
for f in os.listdir(OUTPUT_DIR):
    print(f"  - {f}")